# Multi-*e*GO — mg reference simulation with OpenMM

This notebook runs the **molten-globule (mg) reference simulation** required by the multi-eGO workflow.  
The mg simulation produces the trajectory from which contact probabilities are extracted (via `cmdata`) and used to learn the production force field.

## What you need before starting

From the previous steps you should have:

| File | Description |
|------|-------------|
| `topol_mego.top` | GROMACS topology generated by `mego --system … --egos mg` |
| `ffnonbonded.itp` | Non-bonded parameters (included by the topology) |
| `system.gro` | Starting coordinates (from `gmx pdb2gmx` + energy minimisation) |

## Simulation protocol (matches `ff_aa.mdp`)

1. **Energy minimisation** — steepest descents until max force < 500 kJ/mol/nm  
2. **NVT production** — Langevin dynamics, 300 K, 5 fs timestep, 2.5 nm LJ cutoff, no constraints  

> **Note:** This notebook uses [OpenMM](https://openmm.org) and [ParmEd](https://parmed.github.io/ParmEd/) to load and run the GROMACS topology.  The multi-eGO force field uses C6/C12 non-bonded parameters and no electrostatics.  If you encounter loading errors, please open an issue on the [multi-eGO GitHub](https://github.com/multi-ego/multi-eGO).

## 1 · Install dependencies

In [ ]:
# Install OpenMM, ParmEd, and MDTraj (for trajectory conversion)
!pip install -q openmm parmed mdtraj matplotlib

## 2 · Upload input files

Upload `topol_mego.top`, `ffnonbonded.itp`, and `system.gro` when prompted.

In [ ]:
from google.colab import files
import os

print("Upload topol_mego.top, ffnonbonded.itp, and system.gro")
uploaded = files.upload()

required = {'topol_mego.top', 'ffnonbonded.itp', 'system.gro'}
missing  = required - set(uploaded.keys())
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")
print("\nAll required files present ✓")

## 3 · Simulation parameters

Edit the values in this cell to match your system.

In [ ]:
# ── Simulation parameters ────────────────────────────────────────────────────
TEMPERATURE_K      = 300       # K   — reference temperature
FRICTION_PER_PS    = 0.04      # ps⁻¹ — matches tau_t = 25 ps in ff_aa.mdp
TIMESTEP_FS        = 5.0       # fs
CUTOFF_NM          = 2.5       # nm  — LJ cutoff (same as rvdw in ff_aa.mdp)

N_STEPS_PROD       = 2_000_000 # steps — 10 ns at 5 fs/step
REPORT_EVERY       = 10_000    # steps — log/trajectory output frequency
TRAJ_FILE          = 'trajectory.dcd'
LOG_FILE           = 'simulation.log'
CHECKPOINT_FILE    = 'checkpoint.xml'

## 4 · Load topology and build OpenMM system

In [ ]:
import parmed as pmd
import openmm as mm
import openmm.app as app
import openmm.unit as unit

print("Loading topology with ParmEd...")
struct = pmd.load_file('topol_mego.top', xyz='system.gro')

print(f"  Atoms   : {len(struct.atoms)}")
print(f"  Residues: {len(struct.residues)}")
print(f"  Box     : {struct.box}")

print("\nBuilding OpenMM system...")
system = struct.createSystem(
    nonbondedMethod=app.CutoffPeriodic,
    nonbondedCutoff=CUTOFF_NM * unit.nanometer,
    constraints=None,           # multi-eGO uses no bond constraints
    removeCMMotion=True,
    ignoreExternalBonds=True,
)

# Verify: multi-eGO has no charges — confirm electrostatic contribution is zero
for force in system.getForces():
    print(f"  Force: {type(force).__name__}")

print("\nSystem ready ✓")

## 5 · Energy minimisation

In [ ]:
from openmm.app import StateDataReporter, DCDReporter, CheckpointReporter
import sys

integrator = mm.LangevinMiddleIntegrator(
    TEMPERATURE_K    * unit.kelvin,
    FRICTION_PER_PS  / unit.picosecond,
    TIMESTEP_FS      * unit.femtosecond,
)

# Choose GPU if available, otherwise CPU
try:
    platform = mm.Platform.getPlatformByName('CUDA')
    properties = {'CudaPrecision': 'mixed'}
    print("Using CUDA platform")
except Exception:
    try:
        platform = mm.Platform.getPlatformByName('OpenCL')
        properties = {}
        print("Using OpenCL platform")
    except Exception:
        platform = mm.Platform.getPlatformByName('CPU')
        properties = {}
        print("Using CPU platform")

simulation = app.Simulation(struct.topology, system, integrator, platform, properties)
simulation.context.setPositions(struct.positions)

print(f"Initial potential energy: {simulation.context.getState(getEnergy=True).getPotentialEnergy()}")
print("Running energy minimisation...")
simulation.minimizeEnergy(tolerance=500 * unit.kilojoule_per_mole / unit.nanometer)
print(f"Final   potential energy: {simulation.context.getState(getEnergy=True).getPotentialEnergy()}")
print("Energy minimisation complete ✓")

## 6 · NVT production run

The trajectory is saved as `trajectory.dcd` and converted to `.xtc` at the end for compatibility with `cmdata`.

In [ ]:
import time

# Assign velocities at target temperature
simulation.context.setVelocitiesToTemperature(TEMPERATURE_K * unit.kelvin)

# Reporters
simulation.reporters.clear()
simulation.reporters.append(DCDReporter(TRAJ_FILE, REPORT_EVERY))
simulation.reporters.append(
    StateDataReporter(
        LOG_FILE, REPORT_EVERY,
        step=True, time=True,
        potentialEnergy=True, kineticEnergy=True, totalEnergy=True,
        temperature=True, speed=True, progress=True,
        remainingTime=True, totalSteps=N_STEPS_PROD,
        separator='\t',
    )
)
simulation.reporters.append(
    StateDataReporter(
        sys.stdout, REPORT_EVERY * 10,   # less frequent to stdout
        step=True, potentialEnergy=True, temperature=True,
        progress=True, remainingTime=True, totalSteps=N_STEPS_PROD,
    )
)
simulation.reporters.append(CheckpointReporter(CHECKPOINT_FILE, REPORT_EVERY * 100))

print(f"Running {N_STEPS_PROD:,} steps ({N_STEPS_PROD * TIMESTEP_FS / 1e6:.1f} ns)...\n")
t0 = time.time()
simulation.step(N_STEPS_PROD)
elapsed = time.time() - t0
print(f"\nSimulation complete in {elapsed/60:.1f} min ✓")

# Save final state
simulation.saveState('final_state.xml')
final_pos = simulation.context.getState(getPositions=True, enforcePeriodicBox=True).getPositions()
app.PDBFile.writeFile(struct.topology, final_pos, open('final.pdb', 'w'))
print("Final coordinates saved to final.pdb ✓")

## 7 · Energy analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(LOG_FILE, sep='\t', comment='#')
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

time_ns = df['Time (ps)'] / 1000

axes[0].plot(time_ns, df['Potential Energy (kJ/mole)'], color='steelblue', lw=0.8)
axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('Potential energy (kJ/mol)')
axes[0].set_title('Potential Energy')
axes[0].grid(alpha=0.3)

axes[1].plot(time_ns, df['Temperature (K)'], color='tomato', lw=0.8)
axes[1].axhline(TEMPERATURE_K, color='k', ls='--', lw=1, label=f'{TEMPERATURE_K} K')
axes[1].set_xlabel('Time (ns)')
axes[1].set_ylabel('Temperature (K)')
axes[1].set_title('Temperature')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('energy_plot.png', dpi=150)
plt.show()
print(f"Mean temperature : {df['Temperature (K)'].mean():.1f} ± {df['Temperature (K)'].std():.1f} K")
print(f"Mean pot. energy : {df['Potential Energy (kJ/mole)'].mean():.1f} kJ/mol")

## 8 · Convert trajectory to XTC (for `cmdata`)

`cmdata` expects a GROMACS `.xtc` trajectory.  MDTraj converts the `.dcd` output here.

In [ ]:
import mdtraj as md

print("Loading DCD trajectory...")
traj = md.load(TRAJ_FILE, top='final.pdb')
print(f"  Frames : {traj.n_frames}")
print(f"  Atoms  : {traj.n_atoms}")

xtc_file = TRAJ_FILE.replace('.dcd', '.xtc')
traj.save_xtc(xtc_file)
print(f"Trajectory saved as {xtc_file} ✓")

## 9 · Download output files

Download the trajectory (`.xtc`) and the energy log.  You will need the `.xtc` file together with a GROMACS `.tpr` run-input file to extract contact matrices with `cmdata`.

In [ ]:
from google.colab import files as colab_files
import os

outputs = [xtc_file, LOG_FILE, 'energy_plot.png', 'final.pdb', CHECKPOINT_FILE]
for f in outputs:
    if os.path.exists(f):
        print(f"Downloading {f}...")
        colab_files.download(f)

## Next steps

With the trajectory in hand:

1. Run `cmdata` on the `.xtc` trajectory (requires a `.tpr` file from GROMACS):
   ```bash
   cmdata -f trajectory.xtc -s topol.tpr
   ```

2. Convert the histograms to contact matrices:
   ```bash
   python tools/make_mat/make_mat.py --histo histo/ ...
   ```

3. Generate the production force field:
   ```bash
   mego --config inputs/SYSTEM/config.yml
   ```

See the [multi-eGO documentation](https://github.com/multi-ego/multi-eGO) for the full workflow.